In [5]:
library(readr)
library(dplyr)
library(tidyverse)
library(janitor)
library(readxl)

In [6]:
strip_version <- function(x) sub("\\.\\d+$", "", x)
# pandas treats comparisons with NaN as False; R returns NA, so coalesce to FALSE
same_sign <- function(z, s) coalesce(sign(z) == s, FALSE)
strong    <- function(z, s) same_sign(z, s) & coalesce(abs(z) > 1.96, FALSE)
 
df <- read_tsv("Ast_eqtl_top_assoc.tsv", show_col_types = FALSE) |>
  mutate(gene_id = strip_version(feature))
 
eg <- df |> filter(qval < 0.05)

# eg

In [7]:
# genes that appear in multiple studies (there are 2573)
eg <- eg |>
  mutate(
    n_cohorts  = rowSums(!is.na(pick(z_tissue_0, z_tissue_1, z_tissue_2, z_tissue_3))),
    .s         = sign(Random_Z),
    .rosmap_ok = strong(z_tissue_0, .s) | strong(z_tissue_1, .s),
    .gabitto_ok = strong(z_tissue_2, .s),
    .bryois_ok = strong(z_tissue_3, .s),
    n_blocks_supporting = .rosmap_ok + .gabitto_ok + .bryois_ok,
    n_blocks_concordant_sign =
      (same_sign(z_tissue_0, .s) | same_sign(z_tissue_1, .s)) +
      same_sign(z_tissue_2, .s) + same_sign(z_tissue_3, .s)
  ) |>
  select(-starts_with("."))
 
core <- eg |> filter(n_blocks_supporting >= 2)

# core

In [8]:
tss <- read_tsv("tss_one_base.bed", col_names = c("chrom", "start0", "end0", "gene_id", "score", "strand"),
                col_types = "cddccc") |>
  mutate(gene_id = strip_version(gene_id))
 
eg <- core |> inner_join(select(tss, gene_id, chrom, start0, strand), by = "gene_id")
stopifnot(all(gsub("chr", "", as.character(eg$chr)) == gsub("chr", "", eg$chrom)))

In [9]:
eg <- eg |> mutate(dist_to_tss = (pos - 1) - start0)   # pos is 1-based, start0 is 0-based
HALF <- 524288                                          # half of 1,048,576-bp model input
distal <- eg |> filter(abs(dist_to_tss) > 5000, abs(dist_to_tss) < HALF - 1000)
print(n_distinct(distal$gene_id))

[1] 1977


In [10]:
d <- abs(distal$dist_to_tss)
print(c(count = length(d), mean = mean(d), sd = sd(d), quantile(d)))   # ~ pandas describe()
print(mean(coalesce(abs(eg$dist_to_tss) <= 5000, FALSE)))             # fraction removed by the filter

   count     mean       sd       0%      25%      50%      75%     100% 
  1977.0 103591.0 119002.4   5081.0  22205.0  54537.0 135318.0 522824.0 
[1] 0.1228138


In [11]:
other <- list()
lead  <- list()
for (ct in c("End", "Ext", "IN", "MG", "OD", "OPC")) {
  t <- read_tsv(sprintf("cell_eqtls/%s_eqtl_top_assoc.tsv.gz", ct), show_col_types = FALSE) |>
    mutate(gene_id = strip_version(feature)) |>
    filter(qval < 0.05)
  other[[ct]] <- unique(t$gene_id)
  t <- t[!duplicated(t$gene_id, fromLast = TRUE), ]   # dict(zip(...)) keeps the last duplicate
  lead[[ct]] <- setNames(t$pos, t$gene_id)
}

In [12]:
# genes that are only in astrocytes

distal$n_other_celltypes <- Reduce(`+`, lapply(other, function(s) distal$gene_id %in% s))
ast_specific <- distal |> filter(n_other_celltypes == 0)
print(n_distinct(ast_specific$gene_id))
print(table(distal$n_other_celltypes))

[1] 55

  0   1   2   3   4   5   6 
 55  51 113 189 433 791 345 


In [13]:
# genes that have unique signals in astrocytes
window <- 10000
distal$n_other_same_signal <- Reduce(`+`, lapply(lead, function(l) {
  coalesce(abs(distal$pos - unname(l[distal$gene_id])) <= window, FALSE)
}))
print(sum(distal$n_other_same_signal == 0))

[1] 1190


In [14]:
# distal

In [15]:
# PD risk genes (the 125)
path <- "/home/dh2226/perturb_enrich/data/media-1.xlsx"
hdr <- read_excel(path, sheet = 6, col_names = FALSE,
                  skip = 1, n_max = 2, col_types = "text")   # rows 2-3

group <- as.character(unlist(hdr[1, ])) |> zoo::na.locf(na.rm = FALSE)
field <- as.character(unlist(hdr[2, ]))

# Normalize to catch "GABA_neurons" vs "GABA neurons", "Endothelial cells" vs "Endothelial Cells"
norm <- function(x) str_to_lower(str_replace_all(x, "[_\\s]+", " ")) |> str_trim()
cell_types <- c("da neurons", "glu neurons", "gaba neurons", "astrocytes",
                "oligodendrocytes", "microglia", "opcs", "endothelial cells", "pericytes",
                "substantia nigra", "bulk cortex")
is_ct <- norm(field) %in% cell_types

prefix <- case_when(
  !is_ct ~ NA_character_,
  str_detect(group, regex("P-value of sn-eQTL", ignore_case = TRUE)) ~ "pval",
  str_detect(group, regex("coloc PP4",          ignore_case = TRUE)) ~ "pp4",
  str_detect(group, regex("HEIDI",              ignore_case = TRUE)) ~ "heidi",
  str_detect(group, regex("FDR in bulk",        ignore_case = TRUE)) ~ "bulkfdr",
  TRUE ~ NA_character_
)

# Suffix the repeated peak-SNP column by its block
field2 <- ifelse(str_detect(field, "peakSNP"),
                 paste(field, ifelse(str_detect(group, "HEIDI"), "heidi", "coloc"), sep = "_"),
                 field)

col_names <- ifelse(is.na(prefix), field2, paste(prefix, norm(field), sep = "__")) |>
  make_clean_names()

risk <- read_excel(path, sheet = 6, col_names = col_names,
                   skip = 3, col_types = "text", na = c("", "NA")) |>
  filter(!is.na(gene_symbol)) |>
  mutate(across(matches("^(pval|pp4|heidi|bulkfdr)_|^(b|se|p)_"),
                ~ as.numeric(str_trim(.x))))

New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`
• `` -> `...34`
• `` -> `...35`
• `` -> `...36`
• `` -> `...37`
• `` -> `...38`
• `` -> `...39`
• `` -> `...40`
• `` -> `...41`
• `` -> `...42`
• `` -> `...43`
• `` -> `...44`
• `` -> `...45`
• `` -> `...46`
• `` -> `...47`
• `` -> `...48`
• `` -> `...49`
• `` -> `...50`
• `` -> `...51`
• `` -> `...52`
• `` -> `...53`
• `` -> `...54`
• `` -> `...55`
• `` -> `...56`


In [16]:
names(risk)                # expect pval_da_neurons ... heidi_pericytes, bulkfdr_substantia_nigra
nrow(risk)                 # should be 125; more means footnote rows slipped in
stopifnot(!any(duplicated(names(risk))))
tail(risk$gene_symbol, 5) 

[1] "gene_full_name"               "gene_symbol"                 
 [3] "cytogenetic_band_ncbi"        "gwas_snp"                    
 [5] "gwas_chr"                     "gwas_pos"                    
 [7] "nearest_gene"                 "pval_da_neurons"             
 [9] "pval_glu_neurons"             "pval_gaba_neurons"           
[11] "pval_astrocytes"              "pval_oligodendrocytes"       
[13] "pval_microglia"               "pval_opcs"                   
[15] "pval_endothelial_cells"       "pval_pericytes"              
[17] "e_qtl_chunk_peak_snp_coloc"   "pp4_da_neurons"              
[19] "pp4_glu_neurons"              "pp4_gaba_neurons"            
[21] "pp4_astrocytes"               "pp4_oligodendrocytes"        
[23] "pp4_microglia"                "pp4_opcs"                    
[25] "pp4_endothelial_cells"        "pp4_pericytes"               
[27] "cell_type_nominated_the_gene" "probe_bp"                    
[29] "top_snp_chr"                  "top_snp"                     
[31] "top_snp_bp"                   "effect_allele"               
[33] "other_allele"                 "effect_allele_frequency"     
[35] "b_gwas"                       "se_gwas"                     
[37] "p_gwas"                       "b_e_qtl"                     
[39] "se_e_qtl"                     "p_e_qtl"                     
[41] "b_smr"                        "se_smr"                      
[43] "p_smr"                        "p_smr_multi"                 
[45] "e_qtl_chunk_peak_snp_heidi"   "heidi_da_neurons"            
[47] "heidi_glu_neurons"            "heidi_gaba_neurons"          
[49] "heidi_astrocytes"             "heidi_oligodendrocytes"      
[51] "heidi_microglia"              "heidi_opcs"                  
[53] "heidi_endothelial_cells"      "heidi_pericytes"             
[55] "bulkfdr_substantia_nigra"     "bulkfdr_bulk_cortex"

[1] 125

[1] "BRIP1"     "PGS1"      "RIT2"      "LINC01630" "LSM7"

In [17]:
risk_long <- risk |>
  select(gene_symbol, matches("^(pval|pp4|heidi)_")) |>
  pivot_longer(-gene_symbol,
               names_to = c(".value", "cell_type"),
               names_pattern = "^(pval|pp4|heidi)_+(.*)$")

names(risk_long)
risk_long |> filter(is.na(cell_type)) |> distinct(cell_type)
risk_long

[1] "gene_symbol" "cell_type"   "pval"        "pp4"         "heidi"

cell_type
<chr>


gene_symbol,cell_type,pval,pp4,heidi
<chr>,<chr>,<dbl>,<dbl>,<dbl>
VAMP4,da_neurons,3.77000e-21,0.597598066,2.78e-02
VAMP4,glu_neurons,3.74000e-08,NA,NA
VAMP4,gaba_neurons,6.90000e-05,NA,NA
VAMP4,astrocytes,1.83000e-19,0.843523171,3.17e-01
VAMP4,oligodendrocytes,1.77000e-14,0.876755788,6.62e-01
VAMP4,microglia,9.94000e-05,NA,NA
VAMP4,opcs,1.23151e-02,NA,NA
VAMP4,endothelial_cells,1.61713e-03,NA,NA
VAMP4,pericytes,4.73838e-01,NA,NA


In [18]:
ct_map <- tribble(
  ~cell_type,           ~ct,
  "da_neurons",         "DA",
  "glu_neurons",        "GLU",
  "gaba_neurons",       "GABA",
  "astrocytes",         "Astrocyte",
  "oligodendrocytes",   "Oligodendrocyte",
  "microglia",          "Microglia",
  "opcs",               "OPC",
  "endothelial_cells",  "Endothelial",
  "pericytes",          "Pericyte"
)

risk_long <- risk_long |>
  left_join(ct_map, by = "cell_type")

In [19]:
library(AnnotationDbi)
library(org.Hs.eg.db)

syms <- unique(risk$gene_symbol)

map <- AnnotationDbi::select(org.Hs.eg.db, keys = syms,
                             keytype = "SYMBOL", columns = "ENSEMBL") |>
  as_tibble()

# Rescue old or renamed symbols through aliases
miss <- map |> filter(is.na(ENSEMBL)) |> pull(SYMBOL)
if (length(miss)) {
  alias_map <- AnnotationDbi::select(org.Hs.eg.db, keys = miss,
                                     keytype = "ALIAS", columns = "ENSEMBL") |>
    rename(SYMBOL = ALIAS)
  map <- bind_rows(filter(map, !is.na(ENSEMBL)), alias_map)
}

# Where one symbol maps to several IDs, prefer the one in your eQTL table
map <- map |>
  filter(!is.na(ENSEMBL)) |>
  mutate(in_eqtl = ENSEMBL %in% eqtl$gene_id) |>
  group_by(SYMBOL) |>
  filter(if (any(in_eqtl)) in_eqtl else TRUE) |>
  ungroup() |>
  select(gene_symbol = SYMBOL, gene_id = ENSEMBL) |>
  distinct()

# Symbols with no Ensembl ID at all
setdiff(syms, map$gene_symbol)
map |> count(gene_symbol) |> filter(n > 1)   # still ambiguous: resolve by hand

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following object is masked from ‘package:lubridate’:

    as.difftime


The following object is masked from ‘package:dplyr’:

    explain


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following object is masked from ‘package:dplyr’:

    combine


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS,

ERROR: Error: object 'ALIAS' not found
